In [ ]:
# ResultArray — Usage Examples

# Demonstrates extracting element stress data from an OP2 file and loading it into a `ResultArray` for efficient arithmetic and comparison.

In [5]:
import sys
sys.path.insert(0, r'.')

import numpy as np
from op2_native.reader import OP2
from femdata import ResultArray, concat, stack

## 1 — Load OP2 files

Two runs of the same mesh (linear SOL 101) — same element IDs, different loads.

In [6]:
sc1 = OP2('run_files/model/cylinder-0000.op2')   # subcase 1 = LC1, subcase 2 = LC2

# solid_stresses() → {subcase_id: DataFrame}
# columns: EID, GRID, SX, SY, SZ, SXY, SYZ, SZX, VM
# GRID == 0 rows are centroid averages; GRID > 0 are corner-node results
ss = sc1.solid_stresses()
print(f"subcases available: {list(ss.keys())}")
print(f"columns           : {list(ss[1].columns)}")
print(f"total rows (sc=1) : {len(ss[1])}")
print()
ss[1].head(6)

subcases available: [1, 2]
columns           : ['EID', 'GRID', 'SX', 'SY', 'SZ', 'SXY', 'SYZ', 'SZX', 'VM']
total rows (sc=1) : 19432



,EID,GRID,SX,SY,SZ,SXY,SYZ,SZX,VM
0,1057,529,1.040503,0.708109,6.140589,-0.065100,-1.214339,-0.065832,5.680332
1,1057,530,1.398657,1.528505,5.840130,-0.120299,-0.945863,0.165263,4.687872
2,1057,531,0.569318,0.198492,5.927414,-0.122098,-0.664266,0.147213,5.680415
3,1057,532,1.368882,1.968752,6.564069,0.027287,-1.415633,0.180548,5.508670
4,1058,447,3.231414,4.421458,10.331136,-0.178357,-2.233163,0.412949,7.677327
5,1058,530,1.126858,1.207533,5.422757,0.036449,-0.997790,0.504604,4.676463


## 2 — Build ResultArrays from centroid rows

Keep only `GRID == 0` (centroid) rows, set `EID` as the index, then select
the stress fields of interest.  `ResultArray` stores the data as a sorted
`int32` index + contiguous `float32` matrix — no pandas overhead during arithmetic.

In [7]:
STRESS_COLS = ['VM', 'SX', 'SY', 'SZ', 'SXY', 'SYZ', 'SZX']

def element_envelope(op2_obj, subcase: int, label: str) -> ResultArray:
    """
    Build a ResultArray of element stresses from one subcase.

    This file uses corner-output-only format (no GRID==0 centroid rows),
    so we take the max per column across all corner nodes of each element —
    a per-element design envelope.  The resulting array has one row per EID.
    """
    df = op2_obj.solid_stresses()[subcase]

    # Centroid rows (GRID==0) if present; otherwise envelope corner nodes
    centroid_rows = df[df['GRID'] == 0]
    if len(centroid_rows) > 0:
        per_elem = centroid_rows.set_index('EID')[STRESS_COLS]
    else:
        # Max absolute value per element across all corner-node rows
        per_elem = (
            df.groupby('EID')[STRESS_COLS]
            .agg(lambda col: col.iloc[col.abs().argmax()])
        )

    return ResultArray.from_dataframe(per_elem, domain='element', label=label)


a = element_envelope(sc1, subcase=1, label='LC1')
b = element_envelope(sc1, subcase=2, label='LC2')

print(a)
print()
print(b)

ResultArray [LC1]
  4858 element(s) × 7 field(s)
  columns : VM, SX, SY, SZ, SXY, SYZ, SZX
  index   : [1057…5914]  dtype=float32

ResultArray [LC2]
  4858 element(s) × 7 field(s)
  columns : VM, SX, SY, SZ, SXY, SYZ, SZX
  index   : [1057…5914]  dtype=float32


## 3 — Arithmetic between subcases

All ops auto-align on the shared element index.  When both arrays cover the same mesh the fast path fires — it's a direct numpy call, no reindexing overhead.

In [8]:
# Difference, sum, scale — all element-wise, auto-aligned
diff   = a - b          # LC1 minus LC2, all stress fields
total  = a + b          # superposition
scaled = a * 1.5        # scale entire result set

print("diff  shape:", diff.shape)
print("total shape:", total.shape)

# Single-column retrieval → plain numpy (zero overhead)
vm_diff = diff['VM']
print(f"\nVM stress difference across {len(vm_diff)} elements")
print(f"  min : {vm_diff.min():.4g}")
print(f"  max : {vm_diff.max():.4g}")
print(f"  mean: {vm_diff.mean():.4g}")

diff  shape: (4858, 7)
total shape: (4858, 7)

VM stress difference across 4858 elements
  min : -88.71
  max : 10.96
  mean: -16.78


## 4 — Selecting fields and entities

Column selection with `[]` returns a plain numpy array (single column) or a
new `ResultArray` (multiple columns).  Entity lookup by EID uses `.loc[eid]`.

In [9]:
# Extract just VM and SX for further work
vm_sx = a[['VM', 'SX']]
print("vm_sx:", vm_sx)
print()

# Look up a single element by EID
first_eid = int(a.index[0])
row = a.loc[first_eid]
print(f"EID {first_eid} stresses: {dict(zip(a.columns, row.tolist()))}")
print()

# Filter to the 10 highest-VM elements
top10_idx = np.argsort(a['VM'])[-10:]          # positions (not EIDs)
top10 = a.iloc[top10_idx]
print("Top 10 VM elements:")
print(top10.to_dataframe()[['VM']].sort_values('VM', ascending=False))

vm_sx: ResultArray [LC1]
  4858 element(s) × 2 field(s)
  columns : VM, SX
  index   : [1057…5914]  dtype=float32

EID 1057 stresses: {'VM': 5.68041467666626, 'SX': 1.3986574411392212, 'SY': 1.9687522649765015, 'SZ': 6.564068794250488, 'SXY': -0.1220976859331131, 'SYZ': -1.415633201599121, 'SZX': 0.18054762482643127}

Top 10 VM elements:
                   VM
element_id           
4350        35.574055
2152        35.021530
1153        33.291714
3065        32.942745
5059        32.461941
3450        32.253296
2040        31.353456
4726        30.976254
3064        30.754147
1854        30.667685


## 5 — Summary statistics

In [10]:
a.describe()

ResultArray  [LC1]  4858 element(s) × 7 field(s)
  col             min             max            mean             std
  -------------------------------------------------------------------
  VM             2.36         35.5741         6.01788         3.82633
  SX         -3.60412         20.6186         1.49207         1.90889
  SY         -2.58492         19.3794          1.4907         1.92019
  SZ           2.2846         30.2762         5.92899         3.18365
  SXY        -4.81982         4.90735      0.00606709        0.576928
  SYZ        -18.6407          15.603      -0.0376457         2.28397
  SZX        -16.6957         15.9455      0.00192232          2.2187
